In [ ]:
import math
from typing import Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Utilities
# -----------------------------
def normalize_adj(edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    """
    Row-normalized adjacency with self-loops (mean aggregator).
    edge_index: [2, E] LongTensor of (src, dst)
    Returns indices (same) and values for a sparse tensor A_hat where each row sums to ~1.
    """
    device = edge_index.device
    src, dst = edge_index
    # Add self-loops
    self_loops = torch.arange(num_nodes, device=device)
    sl = torch.stack([self_loops, self_loops], dim=0)
    ei = torch.cat([edge_index, sl], dim=1)

    # Compute out-degree (row-wise after we build row->col indexing)
    row, col = ei
    deg = torch.bincount(row, minlength=num_nodes).clamp(min=1).float()
    vals = torch.ones(ei.shape[1], device=device) / deg[row]
    return ei, vals


def spmm(indices: torch.Tensor, values: torch.Tensor, m: int, n: int, dense: torch.Tensor) -> torch.Tensor:
    """
    Sparse (indices, values) @ dense, where sparse is m x n and dense is n x d
    Returns m x d
    """
    # torch.sparse API (coalesced COO)
    S = torch.sparse_coo_tensor(indices, values, size=(m, n))
    return torch.sparse.mm(S, dense)


# -----------------------------
# GraphSAGE-like message passing
# -----------------------------
class SAGEConv(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, bias: bool = True):
        super().__init__()
        # "Neighbor mean" linear + skip connection linear
        self.lin_neigh = nn.Linear(in_dim, out_dim, bias=bias)
        self.lin_self = nn.Linear(in_dim, out_dim, bias=False)
        self.bn = nn.BatchNorm1d(out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        x: [N, F]
        edge_index: [2, E] with entries in [0, N)
        """
        N, _ = x.shape
        ei, vals = normalize_adj(edge_index, N)  # mean aggregator with self-loops
        neigh = spmm(ei, vals, N, N, x)         # [N, F]
        h = self.lin_neigh(neigh) + self.lin_self(x)
        h = self.bn(h)
        return F.relu(h)


# -----------------------------
# Node Transition GNN (Recurrent)
# -----------------------------
class NodeTransitionGNN(nn.Module):
    def __init__(
        self,
        in_dim: int,        # node feature size N
        hidden_dim: int,    # hidden state size H
        out_dim: int = None,# output size; defaults to in_dim
        num_mp_layers: int = 2,   # message passing depth per step
        residual_delta: bool = True # predict dx instead of x_{t+1}
    ):
        super().__init__()
        out_dim = in_dim if out_dim is None else out_dim
        self.residual_delta = residual_delta

        # Stack a few SAGE layers to compute a "message" from current features
        mp = []
        dims = [in_dim] + [hidden_dim]*(num_mp_layers-1)
        for i in range(num_mp_layers):
            mp.append(SAGEConv(dims[i], hidden_dim))
        self.mp = nn.ModuleList(mp)

        # Per-node recurrent update
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)

        # Readout: hidden -> output (either delta or next state)
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

        # Optional encoder for initial hidden from x_t
        self.encoder = nn.Linear(in_dim, hidden_dim)

    def forward_step(
        self,
        x_t: torch.Tensor,         # [N, F]
        h_t: torch.Tensor,         # [N, H]
        edge_index: torch.Tensor   # [2, E]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        One transition step: (x_t, h_t) -> (x_{t+1_hat}, h_{t+1})
        """
        msg = x_t
        for layer in self.mp:
            msg = layer(msg, edge_index)  # [N, H]

        h_next = self.gru(msg, h_t)       # [N, H]
        y = self.readout(h_next)          # [N, F]
        x_pred = x_t + y if self.residual_delta else y
        return x_pred, h_next

    def forward_sequence(
        self,
        x_seq: torch.Tensor,       # [T, N, F] – teacher-forced inputs
        edge_index: torch.Tensor,  # [2, E]
        rollout: int = 0           # how many future steps to autoregressively roll
    ):
        """
        Returns predictions for the next step at each t in [0..T-2] (teacher forcing),
        and optionally continues 'rollout' steps beyond T-1 using its own predictions.
        """
        T, N, F = x_seq.shape
        device = x_seq.device
        h = torch.tanh(self.encoder(x_seq[0]))  # [N, H]

        preds = []
        x_curr = x_seq[0]
        # Teacher-forced predictions for t=0..T-2 -> predict x_{t+1}
        for t in range(T - 1):
            x_pred, h = self.forward_step(x_curr, h, edge_index)
            preds.append(x_pred)
            x_curr = x_seq[t + 1]  # teacher-forced

        # Optional rollout beyond the provided sequence
        for _ in range(rollout):
            x_pred, h = self.forward_step(x_curr, h, edge_index)
            preds.append(x_pred)
            x_curr = x_pred

        return torch.stack(preds, dim=0)  # [T-1+rollout, N, F]


# -----------------------------
# Synthetic example & training
# -----------------------------
def make_ring_graph(n: int, device: torch.device):
    """
    Simple undirected ring graph.
    Returns edge_index [2,E].
    """
    src = torch.arange(n, device=device)
    dst = (src + 1) % n
    edges = torch.stack([torch.cat([src, dst]), torch.cat([dst, src])], dim=0)  # undirected
    return edges

def simulate_diffusion(edge_index, x0, steps, alpha=0.3):
    """
    Very simple linear diffusion: x_{t+1} = (1-alpha) x_t + alpha * mean(neighbors ∪ self)
    """
    N = x0.shape[0]
    ei, vals = normalize_adj(edge_index, N)
    xs = [x0]
    x = x0
    for _ in range(steps-1):
        neigh = spmm(ei, vals, N, N, x)
        x = (1 - alpha) * x + alpha * neigh
        xs.append(x)
    return torch.stack(xs, dim=0)  # [T, N, F]

def train_demo():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(7)

    # Graph + data
    N = 32       # nodes
    D = 4        # feature dimension per node
    T = 20       # time steps per sequence
    edge_index = make_ring_graph(N, device)

    # Create synthetic sequences (like diffusion)
    x0 = torch.randn(N, D, device=device)
    x_seq = simulate_diffusion(edge_index, x0, steps=T, alpha=0.25)  # [T, N, F]

    # Model
    model = NodeTransitionGNN(
        in_dim=D, hidden_dim=64, out_dim=F, num_mp_layers=2, residual_delta=True
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    # Training loop (teacher forcing)
    epochs = 400
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        preds = model.forward_sequence(x_seq, edge_index, rollout=0)    # [T-1, N, F]
        target = x_seq[1:]                                              # [T-1, N, F]
        loss = F.mse_loss(preds, target)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        if (ep + 1) % 50 == 0:
            with torch.no_grad():
                # Roll a few steps beyond the given sequence to see stability
                rollout_preds = model.forward_sequence(x_seq, edge_index, rollout=5)
                roll_loss = F.mse_loss(rollout_preds[:T-1], target).item()
            print(f"Epoch {ep+1:4d} | train MSE: {loss.item():.6f} | rollout-check MSE: {roll_loss:.6f}")

    # Example: predict next 5 steps autoregressively
    model.eval()
    with torch.no_grad():
        preds_autoreg = model.forward_sequence(x_seq, edge_index, rollout=5)  # [T-1+5, N, F]
        # Compare last 5 teacher-forced predictions vs ground truth if you have more data.
    return model, edge_index, x_seq, preds_autoreg

if __name__ == "__main__":
    model, edge_index, x_seq, preds = train_demo()


TypeError: randn(): argument 'size' failed to unpack the object at pos 2 with error "type must be tuple of ints,but got module"